In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [97]:
df = pd.read_csv("DateFruit_Dataset.csv")
X = df.drop(columns=["Class"])
y = df["Class"]
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [98]:
y = y.map({"BERHI":0, "SOGAY":1, "DEGLET":2, "DOKOL":3, "IRAQI":4, "ROTANA":5, "SAFAVI":6})

In [99]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [100]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

In [101]:
torch.manual_seed(42)
X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scale, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [102]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [103]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [104]:
#ANN MOdel

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #1st Hidden Layer
            nn.Linear(X_train.shape[1], 64),
            nn.ReLU(),

            #2nd Hidden LAyer
            nn.Linear(64,64),
            nn.ReLU(),

            #3rd Output LAyer
            nn.Linear(64,7)
        )

    def forward(self,x):
        return self.model(x)
        

In [105]:
model = ANN()
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())

In [110]:
#Train Model
epochs = 50
train_loss = []
val_loss = []
best_val_loss = float("inf")

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0
    running_val_loss = 0.0

    for xb, yb in train_loader:
        optimiser.zero_grad()
        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimiser.step()
        running_train_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        for xb, yb in test_loader: 
            output = model(xb)
            loss = criterion(output, yb)
            running_val_loss += loss.item()
    
    epoch_train_loss = running_train_loss/len(train_loader)
    epoch_val_loss = running_val_loss/len(test_loader)
    train_loss.append(epoch_train_loss)
    val_loss.append(epoch_val_loss)

    if epoch_val_loss<best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model_ann_classifier.pt")

    print(f"For epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} validation loss = {epoch_val_loss}")

For epoch 1/50 ==> train loss = 0.06518792522990186 validation loss = 0.14194329331318536
For epoch 2/50 ==> train loss = 0.06616666564798873 validation loss = 0.1572281401604414
For epoch 3/50 ==> train loss = 0.0626238998349594 validation loss = 0.14471634229024252
For epoch 4/50 ==> train loss = 0.06730304941858935 validation loss = 0.16030040134986243
For epoch 5/50 ==> train loss = 0.06745377992806227 validation loss = 0.14050363873442015
For epoch 6/50 ==> train loss = 0.0601720101321521 validation loss = 0.16911313496530056
For epoch 7/50 ==> train loss = 0.06479380903360636 validation loss = 0.14040625529984632
For epoch 8/50 ==> train loss = 0.07027752875633862 validation loss = 0.1481624792019526
For epoch 9/50 ==> train loss = 0.0577032864296242 validation loss = 0.15887526609003544
For epoch 10/50 ==> train loss = 0.06245256348958482 validation loss = 0.14451293957730135
For epoch 11/50 ==> train loss = 0.05915478795118954 validation loss = 0.13994500786066055
For epoch 12/

In [111]:
model.load_state_dict(torch.load("best_model_ann_classifier.pt"))

<All keys matched successfully>

In [123]:
#Evaluation

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        output_test = model(xb)
        _, predicted = torch.max(output_test,1)
        correct += (predicted==yb).sum().item()
        total += yb.size(0)

print(f"accuracy = {correct/total*100} %")

accuracy = 94.44444444444444 %
